In [1]:
from CoordMLP import CoordMLP, save_model, load_model
device = 'cuda'
model = CoordMLP()
model = CoordMLP(neurons=200, embedding_size=240).to(device)

In [2]:
import torch
checkpoint_path = "pcamodel_ep_0_l_0.09626400570573516_std_0.22932938318569182_med_0.03550459444522858_best200neu.pth"
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    #step_size = 10  # Reduce LR every 4 epochs
    #gamma = 1/5  # Reduction factor
    #total_epochs = 40
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5, eta_min=1e-4)
startepoch, model, optimizer,scheduler = load_model(model, optimizer, scheduler,checkpoint_path, device)

Model loaded from pcamodel_ep_0_l_0.09626400570573516_std_0.22932938318569182_med_0.03550459444522858_best200neu.pth, resuming from epoch 0


In [3]:
from dataUDFPointCloud import  load_data
lossval = torch.nn.L1Loss(reduction='none')
pca_path = "PCA_PointCloudUDF_dense_rot_center_cleannoise_all_objaverse_0_1fil.npy"
    
    
meshvertices = "objaverse_preprocess_filtered_results.npz"

npzdatatrain = ["objaverse_geodesic_data_0.npz"]
npzdataval = ["objaverse_geodesic_data_val.npz"]

valid_data = load_data(meshvertices, npzdataval, pca_path,num_workers=8, batch_size = 30384)


--- Loading data from: 'objaverse_geodesic_data_val.npz' ---
Appended 'source' from 'objaverse_geodesic_data_val.npz' with shape: (9818236,)
Appended 'dest' from 'objaverse_geodesic_data_val.npz' with shape: (9818236,)


/notebooks/ObjaverseGeodesic/dataUDFPointCloud.py:90: RuntimeWarning: overflow encountered in multiply
  temp_dist_on_a.append(1.42 * data['dist_on_a'][:, None])


Appended 'dist_on_a' from 'objaverse_geodesic_data_val.npz' with shape: (9818236,)

--- Concatenating collected data ---
Concatenated 'mesh_a' total shape: (9818236,), int32
Concatenated 'sources' total shape: (9818236,), int32
Concatenated 'dests' total shape: (9818236,), int32
Concatenated 'dist_on_a' total shape: (9818236, 1), float32
(9545110,)
(9545110, 1) float32
compute euclidean


100%|██████████| 96/96 [00:00<00:00, 337.31it/s]


float32
(9545110, 1)
Mean distance : 0.90989625
Median distance : 0.8202277
Strange distance below euclidean: 0
Minimum distances on mesh a: -2.2739172e-05 (9545110, 1)
(3235,)


In [5]:
from tqdm import tqdm
model.eval()
import numpy as np

total_items = 0
sum_losses = 0.0
sum_sq_losses = 0.0

all_item_losses = []  # <--- add this for median
meshes_id = []
nobatch = 0
with torch.no_grad():
    for mesh1, source, dest, dist1 in tqdm(valid_data):
        meshes_id.append(mesh1)
        nobatch += 1
        mesh1, source, dest, dist1 = (
            mesh1.to(device, non_blocking=True),
            source.to(device, non_blocking=True),
            dest.to(device, non_blocking=True),
            dist1.to(device, non_blocking=True)
        )

        pred = model(source, dest, mesh1, mesh1)
        target = dist1

        per_item_losses = lossval(pred, target)
        per_item_losses_flat = per_item_losses.view(-1)

        batch_items = per_item_losses_flat.numel()

        if batch_items > 0:
            total_items += batch_items
            sum_losses += per_item_losses_flat.sum()
            sum_sq_losses += (per_item_losses_flat ** 2).sum()

            # Store loss on CPU for median
            all_item_losses.append(per_item_losses_flat.detach().cpu())

    # Convert tensor sums to Python scalars if needed
    sum_losses_scalar = sum_losses.item()
    #sum_sq_losses_scalar = sum_sq_losses.item()


    # Median
    all_item_losses_tensor = torch.cat(all_item_losses)  # 1D
    median_loss = torch.median(all_item_losses_tensor).item()
    print("Val loss (median per-item across epoch): ", median_loss)
    print("Val loss (median per item) after scaling to make mean g.t. distance of 1.0 :", median_loss / 0.90989625)

100%|██████████| 314/314 [01:28<00:00,  3.56it/s]


Val loss (median per-item across epoch):  0.03550463914871216
Val loss (median per item) after scaling to make mean g.t. distance of 1.0 : 0.039020535746478964
